In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os

for file in os.listdir("/content"):
    if file.endswith((".mp4", ".avi", ".mov")):
        print(file)

In [ ]:
pie_video_path = "/content/PIE_20seconds.mp4"

In [ ]:
%cd /content
!git clone https://github.com/aras62/PIE.git

In [ ]:
%cd /content/PIE/annotations
!unzip -q annotations.zip -d /content/PIE/annotations_unzipped

In [ ]:
import os

for root, dirs, files in os.walk("/content/PIE"):
    for file in files:
        if file.endswith(".xml"):
            print(os.path.join(root, file))

In [ ]:
import os

for file in os.listdir("/content"):
    if file.endswith((".mp4", ".avi", ".mov")):
        print(file)

In [ ]:
import cv2
import os

# Check available videos
for file in os.listdir("/content"):
    if file.endswith((".mp4", ".avi", ".mov")):
        print(file)

pie_video_path = "/content/Video_forYolo_20seconds.mp4"

cap = cv2.VideoCapture(pie_video_path)

if not cap.isOpened():
    raise ValueError("Video could not be opened. Check filename/path.")

fps = cap.get(cv2.CAP_PROP_FPS)
frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

if fps == 0:
    raise ValueError("FPS is 0, video path or file may be wrong.")

duration = frames / fps

print("FPS:", fps)
print("Total frames:", frames)
print("Duration seconds:", duration)

cap.release()

In [ ]:
import cv2
import xml.etree.ElementTree as ET

video_path = "/content/Video_forYolo_20seconds.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
cap.release()

print("Cut video frame shape:", frame.shape)  # height, width, channels

tree = ET.parse(xml_path)
root = tree.getroot()

max_x = 0
max_y = 0

for track in root.findall("track"):
    if track.attrib.get("label", "") != "pedestrian":
        continue

    for box in track.findall("box"):
        x2 = float(box.attrib["xbr"])
        y2 = float(box.attrib["ybr"])
        max_x = max(max_x, x2)
        max_y = max(max_y, y2)

print("Max annotation x:", max_x)
print("Max annotation y:", max_y)

In [ ]:
import cv2
import xml.etree.ElementTree as ET
import pandas as pd
from ultralytics import YOLO

video_path = "/content/Video_forYolo_20seconds.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

model = YOLO("yolov8m.pt")

# Search around your estimated start frame
candidate_start_frames = list(range(4800, 6301, 30))

conf = 0.30
frame_stride = 30
iou_threshold = 0.50


def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area

    if union == 0:
        return 0

    return inter_area / union


def load_pie_pedestrian_boxes(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    gt_by_frame = {}

    for track in root.findall("track"):
        label = track.attrib.get("label", "")

        if label != "pedestrian":
            continue

        for box in track.findall("box"):
            if box.attrib.get("outside", "0") == "1":
                continue

            frame = int(box.attrib["frame"])

            x1 = float(box.attrib["xtl"])
            y1 = float(box.attrib["ytl"])
            x2 = float(box.attrib["xbr"])
            y2 = float(box.attrib["ybr"])

            gt_by_frame.setdefault(frame, []).append([x1, y1, x2, y2])

    return gt_by_frame


def match_detections_to_gt(yolo_boxes, gt_boxes, iou_threshold=0.50):
    matched_gt = set()
    true_positives = 0

    for det_box in yolo_boxes:
        best_iou = 0
        best_gt_idx = -1

        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in matched_gt:
                continue

            iou = compute_iou(det_box, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= iou_threshold:
            true_positives += 1
            matched_gt.add(best_gt_idx)

    false_positives = len(yolo_boxes) - true_positives
    false_negatives = len(gt_boxes) - true_positives

    return true_positives, false_positives, false_negatives


# Load PIE annotations once
gt_by_frame = load_pie_pedestrian_boxes(xml_path)

# Step 1: Run YOLO once on sampled frames
cap = cv2.VideoCapture(video_path)

yolo_by_cut_frame = {}
cut_frame_idx = 0

print("Running YOLO once on sampled cut-video frames...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if cut_frame_idx % frame_stride == 0:
        results = model(
            frame,
            classes=[0],
            conf=conf,
            iou=0.5,
            verbose=False
        )

        yolo_boxes = []

        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            yolo_boxes.append([float(x1), float(y1), float(x2), float(y2)])

        yolo_by_cut_frame[cut_frame_idx] = yolo_boxes
        print(f"Cut frame {cut_frame_idx}: {len(yolo_boxes)} YOLO boxes")

    cut_frame_idx += 1

cap.release()

print("YOLO pre-computation done.")
print("Sampled frames:", len(yolo_by_cut_frame))


# Step 2: Scan possible start frames without rerunning YOLO
alignment_results = []

for start_frame in candidate_start_frames:

    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_gt = 0
    total_yolo = 0
    evaluated_frames = 0

    for cut_frame_idx, yolo_boxes in yolo_by_cut_frame.items():
        original_frame = start_frame + cut_frame_idx
        gt_boxes = gt_by_frame.get(original_frame, [])

        tp, fp, fn = match_detections_to_gt(
            yolo_boxes,
            gt_boxes,
            iou_threshold
        )

        total_tp += tp
        total_fp += fp
        total_fn += fn
        total_gt += len(gt_boxes)
        total_yolo += len(yolo_boxes)
        evaluated_frames += 1

    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp > 0 else 0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    alignment_results.append({
        "start_frame": start_frame,
        "start_time_seconds": round(start_frame / 30, 2),
        "evaluated_frames": evaluated_frames,
        "gt_pedestrians": total_gt,
        "yolo_detections": total_yolo,
        "true_positives": total_tp,
        "false_positives": total_fp,
        "false_negatives": total_fn,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1_score": round(f1, 3)
    })

df_alignment = pd.DataFrame(alignment_results)

df_alignment.sort_values("f1_score", ascending=False).head(10)

In [ ]:
import cv2
import xml.etree.ElementTree as ET
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

video_path = "/content/Video_forYolo_20seconds.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

start_frame = 5550
cut_frame_number = 0
original_annotation_frame = start_frame + cut_frame_number

model = YOLO("yolov8m.pt")

# Read cut video frame
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, cut_frame_number)
ret, frame = cap.read()
cap.release()

if not ret:
    raise ValueError("Could not read cut video frame.")

# Load PIE annotation boxes
tree = ET.parse(xml_path)
root = tree.getroot()

gt_boxes = []

for track in root.findall("track"):
    label = track.attrib.get("label", "")

    if label != "pedestrian":
        continue

    for box in track.findall("box"):
        if int(box.attrib["frame"]) == original_annotation_frame and box.attrib.get("outside", "0") == "0":
            x1 = int(float(box.attrib["xtl"]))
            y1 = int(float(box.attrib["ytl"]))
            x2 = int(float(box.attrib["xbr"]))
            y2 = int(float(box.attrib["ybr"]))
            gt_boxes.append([x1, y1, x2, y2])

# YOLO detections
results = model(
    frame,
    classes=[0],
    conf=0.30,
    iou=0.5,
    verbose=False
)

yolo_boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)

vis = frame.copy()

# Draw PIE annotation boxes in RED
for (x1, y1, x2, y2) in gt_boxes:
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 0, 255), 3)

# Draw YOLO boxes in BLUE
for (x1, y1, x2, y2) in yolo_boxes:
    cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 0, 0), 2)

print("Cut frame:", cut_frame_number)
print("Original annotation frame:", original_annotation_frame)
print("PIE annotation boxes:", len(gt_boxes))
print("YOLO boxes:", len(yolo_boxes))

cv2_imshow(vis)

In [ ]:
import cv2
import os
import xml.etree.ElementTree as ET
import pandas as pd
from ultralytics import YOLO

# =========================
# CUT VIDEO
# =========================
video_path = "/content/Video_forYolo_20seconds.mp4"

# =========================
# CANDIDATE PIE XML FILES
# =========================
candidate_xmls = [
    "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml",
    "/content/PIE/annotations_unzipped/annotations/set04/video_0008_annt.xml",
    "/content/PIE/annotations_unzipped/annotations/set06/video_0008_annt.xml",
    "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml",
    "/content/PIE/annotations_unzipped/annotations/set04/video_0006_annt.xml",
    "/content/PIE/annotations_unzipped/annotations/set06/video_0006_annt.xml",
]

candidate_xmls = [p for p in candidate_xmls if os.path.exists(p)]

print("Candidate XML files:")
for p in candidate_xmls:
    print(p)

# =========================
# SETTINGS
# =========================
model = YOLO("yolov8m.pt")

conf = 0.30
frame_stride = 30
iou_threshold = 0.50

# Search broad range: 0 to 10 minutes at 30 FPS, every 30 frames
candidate_start_frames = list(range(0, 18000, 30))


# =========================
# FUNCTIONS
# =========================
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area
    return inter_area / union if union > 0 else 0


def load_pie_pedestrian_boxes(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    gt_by_frame = {}

    for track in root.findall("track"):
        label = track.attrib.get("label", "")

        if label != "pedestrian":
            continue

        for box in track.findall("box"):
            if box.attrib.get("outside", "0") == "1":
                continue

            frame = int(box.attrib["frame"])

            x1 = float(box.attrib["xtl"])
            y1 = float(box.attrib["ytl"])
            x2 = float(box.attrib["xbr"])
            y2 = float(box.attrib["ybr"])

            gt_by_frame.setdefault(frame, []).append([x1, y1, x2, y2])

    return gt_by_frame


def match_detections_to_gt(yolo_boxes, gt_boxes, iou_threshold=0.50):
    matched_gt = set()
    true_positives = 0

    for det_box in yolo_boxes:
        best_iou = 0
        best_gt_idx = -1

        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in matched_gt:
                continue

            iou = compute_iou(det_box, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= iou_threshold:
            true_positives += 1
            matched_gt.add(best_gt_idx)

    false_positives = len(yolo_boxes) - true_positives
    false_negatives = len(gt_boxes) - true_positives

    return true_positives, false_positives, false_negatives


# =========================
# STEP 1: RUN YOLO ONLY ONCE
# =========================
cap = cv2.VideoCapture(video_path)

yolo_by_cut_frame = {}
cut_frame_idx = 0

print("\nRunning YOLO once on sampled cut-video frames...")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if cut_frame_idx % frame_stride == 0:
        results = model(
            frame,
            classes=[0],
            conf=conf,
            iou=0.5,
            verbose=False
        )

        yolo_boxes = []

        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            yolo_boxes.append([float(x1), float(y1), float(x2), float(y2)])

        yolo_by_cut_frame[cut_frame_idx] = yolo_boxes
        print(f"Cut frame {cut_frame_idx}: {len(yolo_boxes)} YOLO boxes")

    cut_frame_idx += 1

cap.release()

print("YOLO pre-computation done.")
print("Sampled frames:", len(yolo_by_cut_frame))


# =========================
# STEP 2: SEARCH XML + START FRAME
# =========================
all_results = []

for xml_path in candidate_xmls:
    print("\nChecking:", xml_path)

    gt_by_frame = load_pie_pedestrian_boxes(xml_path)

    for start_frame in candidate_start_frames:
        total_tp = 0
        total_fp = 0
        total_fn = 0
        total_gt = 0
        total_yolo = 0
        evaluated_frames = 0

        for cut_frame_idx, yolo_boxes in yolo_by_cut_frame.items():
            original_frame = start_frame + cut_frame_idx
            gt_boxes = gt_by_frame.get(original_frame, [])

            tp, fp, fn = match_detections_to_gt(
                yolo_boxes,
                gt_boxes,
                iou_threshold
            )

            total_tp += tp
            total_fp += fp
            total_fn += fn
            total_gt += len(gt_boxes)
            total_yolo += len(yolo_boxes)
            evaluated_frames += 1

        precision = total_tp / (total_tp + total_fp) if total_tp + total_fp > 0 else 0
        recall = total_tp / (total_tp + total_fn) if total_tp + total_fn > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

        all_results.append({
            "xml_path": xml_path,
            "start_frame": start_frame,
            "start_time_seconds": round(start_frame / 30, 2),
            "evaluated_frames": evaluated_frames,
            "gt_pedestrians": total_gt,
            "yolo_detections": total_yolo,
            "true_positives": total_tp,
            "false_positives": total_fp,
            "false_negatives": total_fn,
            "precision": round(precision, 3),
            "recall": round(recall, 3),
            "f1_score": round(f1, 3)
        })

df_find_pie = pd.DataFrame(all_results)

df_find_pie.sort_values("f1_score", ascending=False).head(20)

In [ ]:
import os

paths_to_check = [
    "/content/pie_annotations/annotations/set03/video_0008_annt.xml",
    "/content/pie_annotations/annotations/set04/video_0008_annt.xml",
    "/content/pie_annotations/annotations/set06/video_0008_annt.xml",
    "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml",
    "/content/PIE/annotations_unzipped/annotations/set04/video_0008_annt.xml",
    "/content/PIE/annotations_unzipped/annotations/set06/video_0008_annt.xml",
]

for p in paths_to_check:
    print(p, "exists:", os.path.exists(p))

In [ ]:
%cd /content/PIE
!bash download_clips.sh

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "video_0008.mp4":
            print(os.path.join(root, file))

In [ ]:
pie_video_path = "/content/PIE_clips/set03/video_0008.mp4"
pie_xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

In [ ]:
import os

pie_video_path = "/content/PIE_clips/set03/video_0008.mp4"
pie_xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

print("Video exists:", os.path.exists(pie_video_path))
print("XML exists:", os.path.exists(pie_xml_path))

In [ ]:
!find /content -name "download_clips.sh"

In [ ]:
%cd /content/PIE/annotations
!bash download_clips.sh

In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "video_0008.mp4":
            print(os.path.join(root, file))

In [ ]:
import xml.etree.ElementTree as ET
from collections import defaultdict

xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

tree = ET.parse(xml_path)
root = tree.getroot()

crossing_frames = defaultdict(int)

for track in root.findall("track"):
    label = track.attrib.get("label", "")

    if label != "pedestrian":
        continue

    for box in track.findall("box"):
        frame = int(box.attrib["frame"])

        # PIE crossing label can be stored as attribute
        crossing = box.attrib.get("cross", box.attrib.get("crossing", ""))

        if crossing in ["crossing", "1"]:
            crossing_frames[frame] += 1

print("Number of crossing frames:", len(crossing_frames))

# Show top frames with most crossing pedestrians
top_frames = sorted(crossing_frames.items(), key=lambda x: x[1], reverse=True)[:20]

for frame, count in top_frames:
    print("Frame:", frame, "| crossing pedestrians:", count, "| time:", round(frame / 30, 2), "sec")

In [ ]:
import xml.etree.ElementTree as ET

xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

tree = ET.parse(xml_path)
root = tree.getroot()

for track in root.findall("track"):
    if track.attrib.get("label", "") == "pedestrian":
        print("TRACK ATTRIBUTES:")
        print(track.attrib)

        print("\nFIRST 5 BOXES:")
        for box in track.findall("box")[:5]:
            print(box.attrib)

        print("\nFIRST BOX CHILDREN:")
        first_box = track.findall("box")[0]
        for child in first_box:
            print(child.tag, child.attrib, child.text)

        break

In [ ]:
import cv2
import numpy as np
import os

cut_video_path = "/content/Video_forYolo_20seconds.mp4"

candidate_videos = [
    "/content/PIE/annotations/PIE_clips/set03/video_0008.mp4",
    "/content/PIE/annotations/PIE_clips/set04/video_0008.mp4",
    "/content/PIE/annotations/PIE_clips/set06/video_0008.mp4",
]

candidate_videos = [p for p in candidate_videos if os.path.exists(p)]

# Read first frame of your cut video
cut_cap = cv2.VideoCapture(cut_video_path)
ret, cut_frame = cut_cap.read()
cut_cap.release()

if not ret:
    raise ValueError("Could not read cut video first frame.")

# Smaller image = much faster
cut_small = cv2.resize(cut_frame, (160, 90))
cut_gray = cv2.cvtColor(cut_small, cv2.COLOR_BGR2GRAY)

def mse(img1, img2):
    return np.mean((img1.astype("float32") - img2.astype("float32")) ** 2)

best_result = {
    "video_path": None,
    "frame": None,
    "time": None,
    "score": float("inf")
}

# Search around 3:04 only
search_start = 5200
search_end = 5900
stride = 10

for video_path in candidate_videos:
    print("Searching:", video_path)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Jump once to start, then read forward
    cap.set(cv2.CAP_PROP_POS_FRAMES, search_start)

    frame_idx = search_start

    while frame_idx <= search_end:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % stride == 0:
            frame_small = cv2.resize(frame, (160, 90))
            frame_gray = cv2.cvtColor(frame_small, cv2.COLOR_BGR2GRAY)

            score = mse(cut_gray, frame_gray)

            if score < best_result["score"]:
                best_result = {
                    "video_path": video_path,
                    "frame": frame_idx,
                    "time": frame_idx / fps,
                    "score": score
                }

        frame_idx += 1

    cap.release()

print("\nBest rough match:")
print(best_result)

In [ ]:
import cv2
import xml.etree.ElementTree as ET
import pandas as pd
from ultralytics import YOLO

video_path = "/content/PIE/annotations/PIE_clips/set03/video_0008.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

start_frame = 5560
num_frames_to_evaluate = 622   # same length as your 20-second cut

model = YOLO("yolov8m.pt")

conf_values = [0.50, 0.40, 0.35, 0.30, 0.25]
frame_stride = 30
iou_threshold = 0.50


def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area
    return inter_area / union if union > 0 else 0


def load_pie_pedestrian_boxes(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    gt_by_frame = {}

    for track in root.findall("track"):
        if track.attrib.get("label", "") != "pedestrian":
            continue

        for box in track.findall("box"):
            if box.attrib.get("outside", "0") == "1":
                continue

            frame = int(box.attrib["frame"])

            x1 = float(box.attrib["xtl"])
            y1 = float(box.attrib["ytl"])
            x2 = float(box.attrib["xbr"])
            y2 = float(box.attrib["ybr"])

            gt_by_frame.setdefault(frame, []).append([x1, y1, x2, y2])

    return gt_by_frame


def match_detections_to_gt(yolo_boxes, gt_boxes, iou_threshold=0.50):
    matched_gt = set()
    true_positives = 0

    for det_box in yolo_boxes:
        best_iou = 0
        best_gt_idx = -1

        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in matched_gt:
                continue

            iou = compute_iou(det_box, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= iou_threshold:
            true_positives += 1
            matched_gt.add(best_gt_idx)

    false_positives = len(yolo_boxes) - true_positives
    false_negatives = len(gt_boxes) - true_positives

    return true_positives, false_positives, false_negatives


gt_by_frame = load_pie_pedestrian_boxes(xml_path)

results_summary = []

for conf in conf_values:
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    local_frame_idx = 0

    total_tp = 0
    total_fp = 0
    total_fn = 0
    evaluated_frames = 0
    total_yolo_detections = 0
    total_gt_boxes = 0

    while local_frame_idx < num_frames_to_evaluate:
        ret, frame = cap.read()
        if not ret:
            break

        original_frame = start_frame + local_frame_idx

        if local_frame_idx % frame_stride == 0:
            gt_boxes = gt_by_frame.get(original_frame, [])

            results = model(
                frame,
                classes=[0],
                conf=conf,
                iou=0.5,
                verbose=False
            )

            yolo_boxes = []

            for box in results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                yolo_boxes.append([float(x1), float(y1), float(x2), float(y2)])

            tp, fp, fn = match_detections_to_gt(
                yolo_boxes,
                gt_boxes,
                iou_threshold=iou_threshold
            )

            total_tp += tp
            total_fp += fp
            total_fn += fn
            total_yolo_detections += len(yolo_boxes)
            total_gt_boxes += len(gt_boxes)
            evaluated_frames += 1

        local_frame_idx += 1

    cap.release()

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    results_summary.append({
        "confidence": conf,
        "evaluated_frames": evaluated_frames,
        "gt_pedestrians": total_gt_boxes,
        "yolo_detections": total_yolo_detections,
        "true_positives": total_tp,
        "false_positives": total_fp,
        "false_negatives": total_fn,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1_score": round(f1, 3)
    })

df_pie_original = pd.DataFrame(results_summary)
df_pie_original

In [ ]:
import os
import matplotlib.pyplot as plt

save_folder = "/content/PIE_results/yolo_confidence_evaluation/"
os.makedirs(save_folder, exist_ok=True)

# Save CSV
csv_path = save_folder + "pie_yolo_confidence_evaluation.csv"
df_pie_original.to_csv(csv_path, index=False)

print("CSV saved here:")
print(csv_path)

In [ ]:
import matplotlib.pyplot as plt
import os

save_folder = "/content/PIE_results/yolo_confidence_evaluation/"
os.makedirs(save_folder, exist_ok=True)

# Clean column names for thesis table
df_pie_clean = df_pie_original.rename(columns={
    "confidence": "Confidence",
    "evaluated_frames": "Frames",
    "gt_pedestrians": "GT pedestrians",
    "yolo_detections": "YOLO detections",
    "true_positives": "TP",
    "false_positives": "FP",
    "false_negatives": "FN",
    "precision": "Precision",
    "recall": "Recall",
    "f1_score": "F1-score"
})

fig, ax = plt.subplots(figsize=(13, 3))
ax.axis("off")

table = ax.table(
    cellText=df_pie_clean.values,
    colLabels=df_pie_clean.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.15, 1.5)

# Bold header
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
    cell.set_linewidth(0.8)

image_path = save_folder + "pie_yolo_confidence_evaluation_table.png"

plt.savefig(image_path, bbox_inches="tight", dpi=300)
plt.show()

print("Table image saved here:")
print(image_path)

In [ ]:
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0008.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"
start_frame = 5560

In [ ]:
!pip install ultralytics -q

import cv2
import os
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from ultralytics import SAM
from google.colab.patches import cv2_imshow

# =============================
# PIE PATHS
# =============================
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0008.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

start_frame = 5560
num_frames_to_evaluate = 622   # same as your 20-second cut
frame_stride = 30              # every 30th frame
max_boxes_per_frame = 5

output_folder = "/content/PIE_results/sam2_annotation_comparison/"
os.makedirs(output_folder, exist_ok=True)

# =============================
# LOAD SAM2
# =============================
sam_model = SAM("sam2.1_s.pt")

# =============================
# HELPER FUNCTIONS
# =============================

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area
    return inter_area / union if union > 0 else 0


def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]


# =============================
# LOAD PIE ANNOTATION BOXES
# =============================

tree = ET.parse(xml_path)
root = tree.getroot()

annotations_by_frame = {}

for track in root.findall("track"):
    label = track.attrib.get("label", "")

    if label != "pedestrian":
        continue

    for box in track.findall("box"):
        if box.attrib.get("outside", "0") == "1":
            continue

        frame_id = int(box.attrib["frame"])

        x1 = float(box.attrib["xtl"])
        y1 = float(box.attrib["ytl"])
        x2 = float(box.attrib["xbr"])
        y2 = float(box.attrib["ybr"])

        box_w = x2 - x1
        box_h = y2 - y1
        area = box_w * box_h

        # remove tiny/unclear boxes
        if box_h < 45:
            continue
        if area < 1000:
            continue

        annotations_by_frame.setdefault(frame_id, []).append([x1, y1, x2, y2])

print("PIE annotation frames loaded:", len(annotations_by_frame))

# =============================
# PROCESS PIE SEGMENT
# =============================

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

rows = []

local_frame_idx = 0
shown_count = 0
max_show = 5

print("Evaluating SAM2 on PIE annotation prompts...")

while local_frame_idx < num_frames_to_evaluate:
    ret, frame = cap.read()
    if not ret:
        break

    original_frame = start_frame + local_frame_idx

    if local_frame_idx % frame_stride == 0:

        boxes = annotations_by_frame.get(original_frame, [])

        boxes = sorted(
            boxes,
            key=lambda b: (b[2] - b[0]) * (b[3] - b[1]),
            reverse=True
        )[:max_boxes_per_frame]

        if len(boxes) > 0:
            temp_path = "/content/temp_pie_sam2_eval.jpg"
            cv2.imwrite(temp_path, frame)

            boxes_np = np.array(boxes)

            sam_results = sam_model(
                temp_path,
                bboxes=boxes_np,
                verbose=False
            )

            if sam_results[0].masks is not None:
                masks = sam_results[0].masks.data.cpu().numpy()
            else:
                masks = []

            print(f"Frame {original_frame}: prompts={len(boxes)}, masks={len(masks)}")

            for i, mask in enumerate(masks):
                if i >= len(boxes):
                    break

                prompt_box = boxes[i]
                prompt_area = (prompt_box[2] - prompt_box[0]) * (prompt_box[3] - prompt_box[1])

                mask_bool = mask.astype(bool)
                mask_area = int(mask_bool.sum())

                mask_bbox = mask_to_bbox(mask_bool)

                if mask_bbox is not None:
                    mask_bbox_iou = compute_iou(mask_bbox, prompt_box)
                else:
                    mask_bbox_iou = 0

                mask_to_box_ratio = mask_area / prompt_area if prompt_area > 0 else 0

                rows.append({
                    "frame": original_frame,
                    "prompt_id": i + 1,
                    "prompt_box_area": round(prompt_area, 2),
                    "mask_area": mask_area,
                    "mask_to_box_ratio": round(mask_to_box_ratio, 3),
                    "mask_bbox_iou_with_prompt_box": round(mask_bbox_iou, 3)
                })

            # show/save first few visual outputs
            if shown_count < max_show:
                visual_path = f"{output_folder}pie_sam2_annotation_frame_{original_frame}.jpg"
                sam_results[0].save(filename=visual_path)

                result_img = cv2.imread(visual_path)
                cv2_imshow(result_img)

                shown_count += 1

        else:
            print(f"Frame {original_frame}: no usable annotation boxes")

    local_frame_idx += 1

cap.release()

df_sam2_pie = pd.DataFrame(rows)

print("Done.")
print("Total PIE SAM2 prompt-mask pairs evaluated:", len(df_sam2_pie))

df_sam2_pie.head()

In [ ]:
sam2_pie_summary = pd.DataFrame([{
    "dataset": "PIE",
    "video": "set03/video_0008",
    "segment_start_frame": start_frame,
    "total_prompt_masks_evaluated": len(df_sam2_pie),
    "average_mask_to_box_ratio": round(df_sam2_pie["mask_to_box_ratio"].mean(), 3),
    "average_mask_bbox_iou_with_prompt_box": round(df_sam2_pie["mask_bbox_iou_with_prompt_box"].mean(), 3),
    "min_mask_bbox_iou": round(df_sam2_pie["mask_bbox_iou_with_prompt_box"].min(), 3),
    "max_mask_bbox_iou": round(df_sam2_pie["mask_bbox_iou_with_prompt_box"].max(), 3)
}])

sam2_pie_summary

In [ ]:
save_folder = "/content/PIE_results/sam2_annotation_comparison/"
os.makedirs(save_folder, exist_ok=True)

df_sam2_pie.to_csv(save_folder + "pie_sam2_detailed_results.csv", index=False)
sam2_pie_summary.to_csv(save_folder + "pie_sam2_summary.csv", index=False)

print("Saved here:")
print(save_folder)

In [ ]:
import matplotlib.pyplot as plt
import os

save_folder = "/content/PIE_results/sam2_annotation_comparison/"
os.makedirs(save_folder, exist_ok=True)

sam2_pie_summary_clean = sam2_pie_summary.rename(columns={
    "dataset": "Dataset",
    "video": "Video",
    "segment_start_frame": "Start frame",
    "total_prompt_masks_evaluated": "Prompt-mask pairs",
    "average_mask_to_box_ratio": "Avg. mask/box ratio",
    "average_mask_bbox_iou_with_prompt_box": "Avg. mask-box IoU",
    "min_mask_bbox_iou": "Min IoU",
    "max_mask_bbox_iou": "Max IoU"
})

fig, ax = plt.subplots(figsize=(15, 2.5))
ax.axis("off")

table = ax.table(
    cellText=sam2_pie_summary_clean.values,
    colLabels=sam2_pie_summary_clean.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.15, 1.6)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
    cell.set_linewidth(0.8)

image_path = save_folder + "pie_sam2_summary_table.png"

plt.savefig(image_path, bbox_inches="tight", dpi=300)
plt.show()

print("Saved table image here:")
print(image_path)

In [ ]:
!pip install ultralytics -q

import cv2
import os
import numpy as np
from ultralytics import YOLO, SAM
from google.colab.patches import cv2_imshow

# =============================
# PIE PATHS
# =============================
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0008.mp4"

output_folder = "/content/PIE_results/yolo_sam2_combined_same_frames/"
os.makedirs(output_folder, exist_ok=True)

# =============================
# SAME SETTINGS AS BEFORE
# =============================
start_frame = 5560
num_frames_to_process = 622
frame_stride = 30

# Best PIE YOLO confidence from your PIE table
yolo_conf = 0.50
yolo_iou = 0.50

# =============================
# MODELS
# =============================
yolo_model = YOLO("yolov8m.pt")
sam_model = SAM("sam2.1_s.pt")

# =============================
# PROCESS SAME FRAMES
# =============================
cap = cv2.VideoCapture(video_path)

sampled_frames = list(range(start_frame, start_frame + num_frames_to_process, frame_stride))

print("Frames to process:")
print(sampled_frames)

shown_count = 0
max_show = 10
saved_count = 0

colors = [
    (50, 255, 50),
    (255, 50, 50),
    (50, 50, 255),
    (255, 255, 50),
    (255, 50, 255),
    (50, 255, 255),
    (255, 165, 50),
    (150, 50, 255),
]

for frame_number in sampled_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()

    if not ret:
        print(f"Could not read frame {frame_number}")
        continue

    # -----------------------------
    # Step 1: YOLO detection
    # -----------------------------
    yolo_results = yolo_model(
        frame,
        classes=[0],
        conf=yolo_conf,
        iou=yolo_iou,
        verbose=False
    )

    boxes = yolo_results[0].boxes.xyxy.cpu().numpy() if yolo_results[0].boxes is not None else []
    confs = yolo_results[0].boxes.conf.cpu().numpy() if yolo_results[0].boxes is not None else []

    output = frame.copy()

    # -----------------------------
    # Step 2: SAM2 segmentation using YOLO boxes
    # -----------------------------
    if len(boxes) > 0:
        temp_path = "/content/temp_pie_yolo_sam2.jpg"
        cv2.imwrite(temp_path, frame)

        sam_results = sam_model(
            temp_path,
            bboxes=np.array(boxes),
            verbose=False
        )

        masks = sam_results[0].masks.data.cpu().numpy() if sam_results[0].masks is not None else []

        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = box.astype(int)
            color = colors[i % len(colors)]

            # Draw SAM2 mask
            if i < len(masks):
                mask_bool = masks[i].astype(bool)
                colored = output.copy()
                colored[mask_bool] = color
                output = cv2.addWeighted(output, 0.7, colored, 0.3, 0)

            # Draw YOLO box
            cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)

            # Label
            conf_text = f"{confs[i]:.2f}" if i < len(confs) else ""
            label = f"P{i+1} {conf_text}"

            cv2.putText(
                output,
                label,
                (x1, max(y1 - 8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

    save_path = f"{output_folder}pie_yolo_sam2_frame_{frame_number}.jpg"
    cv2.imwrite(save_path, output)
    saved_count += 1

    print(f"Frame {frame_number}: YOLO detections = {len(boxes)}")

    if shown_count < max_show:
        cv2_imshow(output)
        shown_count += 1

cap.release()

print("\nDone.")
print("Saved frames:", saved_count)
print("Output folder:", output_folder)

In [ ]:
!pip install ultralytics -q

import cv2
import os
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from ultralytics import YOLO, SAM
from google.colab.patches import cv2_imshow

# =============================
# PIE PATHS
# =============================
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0008.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0008_annt.xml"

output_folder = "/content/PIE_results/yolo_sam2_combined_with_annotations/"
os.makedirs(output_folder, exist_ok=True)

# =============================
# SAME SETTINGS AS BEFORE
# =============================
start_frame = 5560
num_frames_to_process = 622
frame_stride = 30

# Best PIE YOLO confidence from your table
yolo_conf = 0.50
yolo_iou = 0.50
match_iou_threshold = 0.50

# =============================
# MODELS
# =============================
yolo_model = YOLO("yolov8m.pt")
sam_model = SAM("sam2.1_s.pt")

# =============================
# HELPER FUNCTIONS
# =============================
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area
    return inter_area / union if union > 0 else 0


def match_yolo_to_gt(yolo_boxes, gt_boxes, iou_threshold=0.50):
    matched_gt = set()
    matches = []
    fp_indices = []

    for yolo_idx, yolo_box in enumerate(yolo_boxes):
        best_iou = 0
        best_gt_idx = -1

        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in matched_gt:
                continue

            iou = compute_iou(yolo_box, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= iou_threshold:
            matches.append((yolo_idx, best_gt_idx, best_iou))
            matched_gt.add(best_gt_idx)
        else:
            fp_indices.append(yolo_idx)

    fn_indices = [i for i in range(len(gt_boxes)) if i not in matched_gt]

    return matches, fp_indices, fn_indices


# =============================
# LOAD PIE ANNOTATIONS
# =============================
tree = ET.parse(xml_path)
root = tree.getroot()

gt_by_frame = {}

for track in root.findall("track"):
    label = track.attrib.get("label", "")

    if label != "pedestrian":
        continue

    for box in track.findall("box"):
        if box.attrib.get("outside", "0") == "1":
            continue

        frame_id = int(box.attrib["frame"])

        x1 = float(box.attrib["xtl"])
        y1 = float(box.attrib["ytl"])
        x2 = float(box.attrib["xbr"])
        y2 = float(box.attrib["ybr"])

        gt_by_frame.setdefault(frame_id, []).append([x1, y1, x2, y2])

print("Loaded PIE annotations.")

# =============================
# PROCESS SAME FRAMES
# =============================
cap = cv2.VideoCapture(video_path)

sampled_frames = list(range(start_frame, start_frame + num_frames_to_process, frame_stride))

print("Frames to process:")
print(sampled_frames)

shown_count = 0
max_show = 10
saved_count = 0

summary_rows = []

colors = [
    (50, 255, 50),
    (255, 50, 50),
    (50, 50, 255),
    (255, 255, 50),
    (255, 50, 255),
    (50, 255, 255),
    (255, 165, 50),
    (150, 50, 255),
]

for frame_number in sampled_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()

    if not ret:
        print(f"Could not read frame {frame_number}")
        continue

    gt_boxes = gt_by_frame.get(frame_number, [])

    # -----------------------------
    # Step 1: YOLO detection
    # -----------------------------
    yolo_results = yolo_model(
        frame,
        classes=[0],
        conf=yolo_conf,
        iou=yolo_iou,
        verbose=False
    )

    boxes = yolo_results[0].boxes.xyxy.cpu().numpy() if yolo_results[0].boxes is not None else []
    confs = yolo_results[0].boxes.conf.cpu().numpy() if yolo_results[0].boxes is not None else []

    output = frame.copy()

    # -----------------------------
    # Step 2: SAM2 segmentation using YOLO boxes
    # -----------------------------
    if len(boxes) > 0:
        temp_path = "/content/temp_pie_yolo_sam2_annotated.jpg"
        cv2.imwrite(temp_path, frame)

        sam_results = sam_model(
            temp_path,
            bboxes=np.array(boxes),
            verbose=False
        )

        masks = sam_results[0].masks.data.cpu().numpy() if sam_results[0].masks is not None else []

        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = box.astype(int)
            color = colors[i % len(colors)]

            # Draw SAM2 mask
            if i < len(masks):
                mask_bool = masks[i].astype(bool)
                colored = output.copy()
                colored[mask_bool] = color
                output = cv2.addWeighted(output, 0.7, colored, 0.3, 0)

            # Draw YOLO box
            cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)

            # Label
            conf_text = f"{confs[i]:.2f}" if i < len(confs) else ""
            label = f"P{i+1} {conf_text}"

            cv2.putText(
                output,
                label,
                (x1, max(y1 - 8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

    # -----------------------------
    # Step 3: Draw annotation boxes on top
    # -----------------------------
    for gt_box in gt_boxes:
        x1, y1, x2, y2 = map(int, gt_box)
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 0, 255), 3)

    # -----------------------------
    # Step 4: Compare YOLO boxes with annotations
    # -----------------------------
    matches, fp_indices, fn_indices = match_yolo_to_gt(
        boxes,
        gt_boxes,
        iou_threshold=match_iou_threshold
    )

    tp = len(matches)
    fp = len(fp_indices)
    fn = len(fn_indices)

    summary_rows.append({
        "frame": frame_number,
        "gt_annotations": len(gt_boxes),
        "yolo_detections": len(boxes),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn
    })

    # Add small info text
    info = f"GT: {len(gt_boxes)} | YOLO: {len(boxes)} | TP: {tp} FP: {fp} FN: {fn}"
    cv2.putText(
        output,
        info,
        (30, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 0, 255),
        3
    )

    save_path = f"{output_folder}pie_yolo_sam2_annotated_frame_{frame_number}.jpg"
    cv2.imwrite(save_path, output)
    saved_count += 1

    print(f"Frame {frame_number}: GT={len(gt_boxes)}, YOLO={len(boxes)}, TP={tp}, FP={fp}, FN={fn}")

    if shown_count < max_show:
        cv2_imshow(output)
        shown_count += 1

cap.release()

df_pie_combined_annotation = pd.DataFrame(summary_rows)

print("\nDone.")
print("Saved frames:", saved_count)
print("Output folder:", output_folder)

df_pie_combined_annotation

In [ ]:
csv_path = "/content/PIE_results/yolo_sam2_combined_with_annotations/pie_combined_annotation_comparison.csv"
df_pie_combined_annotation.to_csv(csv_path, index=False)

print("Saved table here:")
print(csv_path)

In [ ]:
tp_total = df_pie_combined_annotation["true_positives"].sum()
fp_total = df_pie_combined_annotation["false_positives"].sum()
fn_total = df_pie_combined_annotation["false_negatives"].sum()

gt_total = df_pie_combined_annotation["gt_annotations"].sum()
yolo_total = df_pie_combined_annotation["yolo_detections"].sum()

precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

combined_summary = pd.DataFrame([{
    "dataset": "PIE",
    "video": "set03/video_0008",
    "start_frame": 5560,
    "sampled_frames": len(df_pie_combined_annotation),
    "gt_annotations": gt_total,
    "yolo_detections": yolo_total,
    "true_positives": tp_total,
    "false_positives": fp_total,
    "false_negatives": fn_total,
    "precision": round(precision, 3),
    "recall": round(recall, 3),
    "f1_score": round(f1, 3)
}])

combined_summary

In [ ]:
import os

save_folder = "/content/PIE_results/yolo_sam2_combined_with_annotations/"
os.makedirs(save_folder, exist_ok=True)

df_pie_combined_annotation.to_csv(
    save_folder + "pie_yolo_sam2_framewise_annotation_comparison.csv",
    index=False
)

combined_summary.to_csv(
    save_folder + "pie_yolo_sam2_annotation_summary.csv",
    index=False
)

print("Saved here:")
print(save_folder)

In [ ]:
import matplotlib.pyplot as plt
import os

save_folder = "/content/PIE_results/yolo_sam2_combined_with_annotations/"
os.makedirs(save_folder, exist_ok=True)

# Clean column names
combined_summary_clean = combined_summary.rename(columns={
    "dataset": "Dataset",
    "video": "Video",
    "start_frame": "Start frame",
    "sampled_frames": "Frames",
    "gt_annotations": "GT annotations",
    "yolo_detections": "YOLO detections",
    "true_positives": "TP",
    "false_positives": "FP",
    "false_negatives": "FN",
    "precision": "Precision",
    "recall": "Recall",
    "f1_score": "F1-score"
})

fig, ax = plt.subplots(figsize=(15, 2.5))
ax.axis("off")

table = ax.table(
    cellText=combined_summary_clean.values,
    colLabels=combined_summary_clean.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.15, 1.7)

# Bold header
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
    cell.set_linewidth(0.8)

image_path = save_folder + "pie_yolo_sam2_combined_summary_table.png"

plt.savefig(image_path, bbox_inches="tight", dpi=300)
plt.show()

print("Saved table image here:")
print(image_path)

In [ ]:
import os

video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

print("Video exists:", os.path.exists(video_path))
print("XML exists:", os.path.exists(xml_path))

In [ ]:
import xml.etree.ElementTree as ET
from collections import defaultdict

xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

tree = ET.parse(xml_path)
root = tree.getroot()

crossing_frames = defaultdict(int)

for track in root.findall("track"):
    if track.attrib.get("label", "") != "pedestrian":
        continue

    for box in track.findall("box"):
        frame = int(box.attrib["frame"])

        cross_value = None

        for child in box:
            if child.tag == "attribute" and child.attrib.get("name") == "cross":
                cross_value = child.text

        if cross_value == "crossing":
            crossing_frames[frame] += 1

print("Number of crossing frames:", len(crossing_frames))

top_frames = sorted(crossing_frames.items(), key=lambda x: x[1], reverse=True)[:30]

for frame, count in top_frames:
    print("Frame:", frame, "| crossing pedestrians:", count, "| time:", round(frame / 30, 2), "sec")

In [ ]:
center_frame = 7170
start_frame = 6870
num_frames_to_process = 622

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"

frame_number = 7170

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
ret, frame = cap.read()
cap.release()

if ret:
    print("Showing frame:", frame_number)
    cv2_imshow(frame)
else:
    print("Could not read frame")

In [ ]:
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

center_frame = 7170
start_frame = 6870
num_frames_to_process = 622
frame_stride = 30

In [ ]:
!pip install ultralytics -q

import cv2
import xml.etree.ElementTree as ET
import pandas as pd
import os
from ultralytics import YOLO

video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

start_frame = 6870
num_frames_to_evaluate = 622

model = YOLO("yolov8m.pt")

conf_values = [0.50, 0.40, 0.35, 0.30, 0.25]
frame_stride = 30
iou_threshold = 0.50

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area
    return inter_area / union if union > 0 else 0

def load_pie_pedestrian_boxes(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    gt_by_frame = {}

    for track in root.findall("track"):
        if track.attrib.get("label", "") != "pedestrian":
            continue

        for box in track.findall("box"):
            if box.attrib.get("outside", "0") == "1":
                continue

            frame = int(box.attrib["frame"])

            x1 = float(box.attrib["xtl"])
            y1 = float(box.attrib["ytl"])
            x2 = float(box.attrib["xbr"])
            y2 = float(box.attrib["ybr"])

            gt_by_frame.setdefault(frame, []).append([x1, y1, x2, y2])

    return gt_by_frame

def match_detections_to_gt(yolo_boxes, gt_boxes, iou_threshold=0.50):
    matched_gt = set()
    true_positives = 0

    for det_box in yolo_boxes:
        best_iou = 0
        best_gt_idx = -1

        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in matched_gt:
                continue

            iou = compute_iou(det_box, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= iou_threshold:
            true_positives += 1
            matched_gt.add(best_gt_idx)

    false_positives = len(yolo_boxes) - true_positives
    false_negatives = len(gt_boxes) - true_positives

    return true_positives, false_positives, false_negatives

gt_by_frame = load_pie_pedestrian_boxes(xml_path)

results_summary = []

for conf in conf_values:
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    local_frame_idx = 0

    total_tp = 0
    total_fp = 0
    total_fn = 0
    evaluated_frames = 0
    total_yolo_detections = 0
    total_gt_boxes = 0

    while local_frame_idx < num_frames_to_evaluate:
        ret, frame = cap.read()
        if not ret:
            break

        original_frame = start_frame + local_frame_idx

        if local_frame_idx % frame_stride == 0:
            gt_boxes = gt_by_frame.get(original_frame, [])

            results = model(
                frame,
                classes=[0],
                conf=conf,
                iou=0.5,
                verbose=False
            )

            yolo_boxes = []

            for box in results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                yolo_boxes.append([float(x1), float(y1), float(x2), float(y2)])

            tp, fp, fn = match_detections_to_gt(
                yolo_boxes,
                gt_boxes,
                iou_threshold=iou_threshold
            )

            total_tp += tp
            total_fp += fp
            total_fn += fn
            total_yolo_detections += len(yolo_boxes)
            total_gt_boxes += len(gt_boxes)
            evaluated_frames += 1

        local_frame_idx += 1

    cap.release()

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    results_summary.append({
        "confidence": conf,
        "evaluated_frames": evaluated_frames,
        "gt_pedestrians": total_gt_boxes,
        "yolo_detections": total_yolo_detections,
        "true_positives": total_tp,
        "false_positives": total_fp,
        "false_negatives": total_fn,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1_score": round(f1, 3)
    })

df_pie2_yolo = pd.DataFrame(results_summary)
df_pie2_yolo

In [ ]:
import os
import matplotlib.pyplot as plt

save_folder = "/content/PIE_results/video_0006_yolo_evaluation/"
os.makedirs(save_folder, exist_ok=True)

# Save CSV
csv_path = save_folder + "pie_video_0006_yolo_confidence_evaluation.csv"
df_pie2_yolo.to_csv(csv_path, index=False)

# Clean column names
df_pie2_clean = df_pie2_yolo.rename(columns={
    "confidence": "Confidence",
    "evaluated_frames": "Frames",
    "gt_pedestrians": "GT pedestrians",
    "yolo_detections": "YOLO detections",
    "true_positives": "TP",
    "false_positives": "FP",
    "false_negatives": "FN",
    "precision": "Precision",
    "recall": "Recall",
    "f1_score": "F1-score"
})

fig, ax = plt.subplots(figsize=(13, 3))
ax.axis("off")

table = ax.table(
    cellText=df_pie2_clean.values,
    colLabels=df_pie2_clean.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.15, 1.5)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
    cell.set_linewidth(0.8)

image_path = save_folder + "pie_video_0006_yolo_confidence_table.png"

plt.savefig(image_path, bbox_inches="tight", dpi=300)
plt.show()

print("Saved CSV:")
print(csv_path)
print("Saved table image:")
print(image_path)

In [ ]:
!pip install ultralytics -q

import cv2
import os
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from ultralytics import SAM
from google.colab.patches import cv2_imshow

# =============================
# PIE VIDEO 2 PATHS
# =============================
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

start_frame = 6870
num_frames_to_evaluate = 622
frame_stride = 30
max_boxes_per_frame = 5

output_folder = "/content/PIE_results/video_0006_sam2_annotation_comparison/"
os.makedirs(output_folder, exist_ok=True)

# =============================
# LOAD SAM2
# =============================
sam_model = SAM("sam2.1_s.pt")

# =============================
# HELPER FUNCTIONS
# =============================
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area
    return inter_area / union if union > 0 else 0


def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]


# =============================
# LOAD PIE ANNOTATION BOXES
# =============================
tree = ET.parse(xml_path)
root = tree.getroot()

annotations_by_frame = {}

for track in root.findall("track"):
    label = track.attrib.get("label", "")

    if label != "pedestrian":
        continue

    for box in track.findall("box"):
        if box.attrib.get("outside", "0") == "1":
            continue

        frame_id = int(box.attrib["frame"])

        x1 = float(box.attrib["xtl"])
        y1 = float(box.attrib["ytl"])
        x2 = float(box.attrib["xbr"])
        y2 = float(box.attrib["ybr"])

        box_w = x2 - x1
        box_h = y2 - y1
        area = box_w * box_h

        # remove tiny/unclear boxes
        if box_h < 45:
            continue
        if area < 1000:
            continue

        annotations_by_frame.setdefault(frame_id, []).append([x1, y1, x2, y2])

print("PIE video 2 annotation frames loaded:", len(annotations_by_frame))

# =============================
# PROCESS SELECTED SEGMENT
# =============================
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

rows = []

local_frame_idx = 0
shown_count = 0
max_show = 5

print("Evaluating SAM2 on PIE video 2 annotation prompts...")

while local_frame_idx < num_frames_to_evaluate:
    ret, frame = cap.read()
    if not ret:
        break

    original_frame = start_frame + local_frame_idx

    if local_frame_idx % frame_stride == 0:

        boxes = annotations_by_frame.get(original_frame, [])

        boxes = sorted(
            boxes,
            key=lambda b: (b[2] - b[0]) * (b[3] - b[1]),
            reverse=True
        )[:max_boxes_per_frame]

        if len(boxes) > 0:
            temp_path = "/content/temp_pie2_sam2_eval.jpg"
            cv2.imwrite(temp_path, frame)

            boxes_np = np.array(boxes)

            sam_results = sam_model(
                temp_path,
                bboxes=boxes_np,
                verbose=False
            )

            if sam_results[0].masks is not None:
                masks = sam_results[0].masks.data.cpu().numpy()
            else:
                masks = []

            print(f"Frame {original_frame}: prompts={len(boxes)}, masks={len(masks)}")

            for i, mask in enumerate(masks):
                if i >= len(boxes):
                    break

                prompt_box = boxes[i]
                prompt_area = (prompt_box[2] - prompt_box[0]) * (prompt_box[3] - prompt_box[1])

                mask_bool = mask.astype(bool)
                mask_area = int(mask_bool.sum())

                mask_bbox = mask_to_bbox(mask_bool)

                if mask_bbox is not None:
                    mask_bbox_iou = compute_iou(mask_bbox, prompt_box)
                else:
                    mask_bbox_iou = 0

                mask_to_box_ratio = mask_area / prompt_area if prompt_area > 0 else 0

                rows.append({
                    "frame": original_frame,
                    "prompt_id": i + 1,
                    "prompt_box_area": round(prompt_area, 2),
                    "mask_area": mask_area,
                    "mask_to_box_ratio": round(mask_to_box_ratio, 3),
                    "mask_bbox_iou_with_prompt_box": round(mask_bbox_iou, 3)
                })

            # Show/save first few visual outputs
            if shown_count < max_show:
                visual_path = f"{output_folder}pie2_sam2_annotation_frame_{original_frame}.jpg"
                sam_results[0].save(filename=visual_path)

                result_img = cv2.imread(visual_path)
                cv2_imshow(result_img)

                shown_count += 1

        else:
            print(f"Frame {original_frame}: no usable annotation boxes")

    local_frame_idx += 1

cap.release()

df_sam2_pie2 = pd.DataFrame(rows)

print("Done.")
print("Total PIE video 2 SAM2 prompt-mask pairs evaluated:", len(df_sam2_pie2))

df_sam2_pie2.head()

In [ ]:
sam2_pie2_summary = pd.DataFrame([{
    "dataset": "PIE",
    "video": "set03/video_0006",
    "segment_start_frame": start_frame,
    "total_prompt_masks_evaluated": len(df_sam2_pie2),
    "average_mask_to_box_ratio": round(df_sam2_pie2["mask_to_box_ratio"].mean(), 3),
    "average_mask_bbox_iou_with_prompt_box": round(df_sam2_pie2["mask_bbox_iou_with_prompt_box"].mean(), 3),
    "min_mask_bbox_iou": round(df_sam2_pie2["mask_bbox_iou_with_prompt_box"].min(), 3),
    "max_mask_bbox_iou": round(df_sam2_pie2["mask_bbox_iou_with_prompt_box"].max(), 3)
}])

sam2_pie2_summary

In [ ]:
import os
import matplotlib.pyplot as plt

save_folder = "/content/PIE_results/video_0006_sam2_annotation_comparison/"
os.makedirs(save_folder, exist_ok=True)

df_sam2_pie2.to_csv(save_folder + "pie_video_0006_sam2_detailed_results.csv", index=False)
sam2_pie2_summary.to_csv(save_folder + "pie_video_0006_sam2_summary.csv", index=False)

sam2_pie2_summary_clean = sam2_pie2_summary.rename(columns={
    "dataset": "Dataset",
    "video": "Video",
    "segment_start_frame": "Start frame",
    "total_prompt_masks_evaluated": "Prompt-mask pairs",
    "average_mask_to_box_ratio": "Avg. mask/box ratio",
    "average_mask_bbox_iou_with_prompt_box": "Avg. mask-box IoU",
    "min_mask_bbox_iou": "Min IoU",
    "max_mask_bbox_iou": "Max IoU"
})

fig, ax = plt.subplots(figsize=(15, 2.5))
ax.axis("off")

table = ax.table(
    cellText=sam2_pie2_summary_clean.values,
    colLabels=sam2_pie2_summary_clean.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.15, 1.6)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
    cell.set_linewidth(0.8)

image_path = save_folder + "pie_video_0006_sam2_summary_table.png"

plt.savefig(image_path, bbox_inches="tight", dpi=300)
plt.show()

print("Saved files here:")
print(save_folder)
print("Saved table image:")
print(image_path)

In [ ]:
!pip install ultralytics -q

import cv2
import os
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from ultralytics import YOLO, SAM
from google.colab.patches import cv2_imshow

# =============================
# PIE VIDEO 2 PATHS
# =============================
video_path = "/content/PIE/annotations/PIE_clips/set03/video_0006.mp4"
xml_path = "/content/PIE/annotations_unzipped/annotations/set03/video_0006_annt.xml"

output_folder = "/content/PIE_results/video_0006_yolo_sam2_combined_with_annotations/"
os.makedirs(output_folder, exist_ok=True)

# =============================
# SETTINGS
# =============================
start_frame = 6870
num_frames_to_process = 622
frame_stride = 30

yolo_conf = 0.50
yolo_iou = 0.50
match_iou_threshold = 0.50

# =============================
# MODELS
# =============================
yolo_model = YOLO("yolov8m.pt")
sam_model = SAM("sam2.1_s.pt")

# =============================
# HELPER FUNCTIONS
# =============================
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = inter_w * inter_h

    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])

    union = areaA + areaB - inter_area
    return inter_area / union if union > 0 else 0


def match_yolo_to_gt(yolo_boxes, gt_boxes, iou_threshold=0.50):
    matched_gt = set()
    matches = []
    fp_indices = []

    for yolo_idx, yolo_box in enumerate(yolo_boxes):
        best_iou = 0
        best_gt_idx = -1

        for gt_idx, gt_box in enumerate(gt_boxes):
            if gt_idx in matched_gt:
                continue

            iou = compute_iou(yolo_box, gt_box)

            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx

        if best_iou >= iou_threshold:
            matches.append((yolo_idx, best_gt_idx, best_iou))
            matched_gt.add(best_gt_idx)
        else:
            fp_indices.append(yolo_idx)

    fn_indices = [i for i in range(len(gt_boxes)) if i not in matched_gt]

    return matches, fp_indices, fn_indices


# =============================
# LOAD PIE ANNOTATIONS
# =============================
tree = ET.parse(xml_path)
root = tree.getroot()

gt_by_frame = {}

for track in root.findall("track"):
    if track.attrib.get("label", "") != "pedestrian":
        continue

    for box in track.findall("box"):
        if box.attrib.get("outside", "0") == "1":
            continue

        frame_id = int(box.attrib["frame"])

        x1 = float(box.attrib["xtl"])
        y1 = float(box.attrib["ytl"])
        x2 = float(box.attrib["xbr"])
        y2 = float(box.attrib["ybr"])

        gt_by_frame.setdefault(frame_id, []).append([x1, y1, x2, y2])

print("Loaded PIE video 2 annotations.")

# =============================
# PROCESS SAME FRAMES
# =============================
cap = cv2.VideoCapture(video_path)

sampled_frames = list(range(start_frame, start_frame + num_frames_to_process, frame_stride))

shown_count = 0
max_show = 10
saved_count = 0

summary_rows = []

colors = [
    (50, 255, 50),
    (255, 50, 50),
    (50, 50, 255),
    (255, 255, 50),
    (255, 50, 255),
    (50, 255, 255),
    (255, 165, 50),
    (150, 50, 255),
]

for frame_number in sampled_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()

    if not ret:
        print(f"Could not read frame {frame_number}")
        continue

    gt_boxes = gt_by_frame.get(frame_number, [])

    # -----------------------------
    # Step 1: YOLO detection
    # -----------------------------
    yolo_results = yolo_model(
        frame,
        classes=[0],
        conf=yolo_conf,
        iou=yolo_iou,
        verbose=False
    )

    boxes = yolo_results[0].boxes.xyxy.cpu().numpy() if yolo_results[0].boxes is not None else []
    confs = yolo_results[0].boxes.conf.cpu().numpy() if yolo_results[0].boxes is not None else []

    output = frame.copy()

    # -----------------------------
    # Step 2: SAM2 segmentation using YOLO boxes
    # -----------------------------
    if len(boxes) > 0:
        temp_path = "/content/temp_pie2_yolo_sam2_annotated.jpg"
        cv2.imwrite(temp_path, frame)

        sam_results = sam_model(
            temp_path,
            bboxes=np.array(boxes),
            verbose=False
        )

        masks = sam_results[0].masks.data.cpu().numpy() if sam_results[0].masks is not None else []

        for i, box in enumerate(boxes):
            x1, y1, x2, y2 = box.astype(int)
            color = colors[i % len(colors)]

            if i < len(masks):
                mask_bool = masks[i].astype(bool)
                colored = output.copy()
                colored[mask_bool] = color
                output = cv2.addWeighted(output, 0.7, colored, 0.3, 0)

            cv2.rectangle(output, (x1, y1), (x2, y2), color, 2)

            conf_text = f"{confs[i]:.2f}" if i < len(confs) else ""
            label = f"P{i+1} {conf_text}"

            cv2.putText(
                output,
                label,
                (x1, max(y1 - 8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

    # -----------------------------
    # Step 3: Draw annotation boxes on top
    # -----------------------------
    for gt_box in gt_boxes:
        x1, y1, x2, y2 = map(int, gt_box)
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 0, 255), 3)

    # -----------------------------
    # Step 4: Compare YOLO boxes with annotations
    # -----------------------------
    matches, fp_indices, fn_indices = match_yolo_to_gt(
        boxes,
        gt_boxes,
        iou_threshold=match_iou_threshold
    )

    tp = len(matches)
    fp = len(fp_indices)
    fn = len(fn_indices)

    summary_rows.append({
        "frame": frame_number,
        "gt_annotations": len(gt_boxes),
        "yolo_detections": len(boxes),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn
    })

    info = f"GT: {len(gt_boxes)} | YOLO: {len(boxes)} | TP: {tp} FP: {fp} FN: {fn}"
    cv2.putText(
        output,
        info,
        (30, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 0, 255),
        3
    )

    save_path = f"{output_folder}pie2_yolo_sam2_annotated_frame_{frame_number}.jpg"
    cv2.imwrite(save_path, output)
    saved_count += 1

    print(f"Frame {frame_number}: GT={len(gt_boxes)}, YOLO={len(boxes)}, TP={tp}, FP={fp}, FN={fn}")

    if shown_count < max_show:
        cv2_imshow(output)
        shown_count += 1

cap.release()

df_pie2_combined_annotation = pd.DataFrame(summary_rows)

print("\nDone.")
print("Saved frames:", saved_count)
print("Output folder:", output_folder)

df_pie2_combined_annotation

In [ ]:
tp_total = df_pie2_combined_annotation["true_positives"].sum()
fp_total = df_pie2_combined_annotation["false_positives"].sum()
fn_total = df_pie2_combined_annotation["false_negatives"].sum()

gt_total = df_pie2_combined_annotation["gt_annotations"].sum()
yolo_total = df_pie2_combined_annotation["yolo_detections"].sum()

precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

pie2_combined_summary = pd.DataFrame([{
    "dataset": "PIE",
    "video": "set03/video_0006",
    "start_frame": start_frame,
    "sampled_frames": len(df_pie2_combined_annotation),
    "gt_annotations": gt_total,
    "yolo_detections": yolo_total,
    "true_positives": tp_total,
    "false_positives": fp_total,
    "false_negatives": fn_total,
    "precision": round(precision, 3),
    "recall": round(recall, 3),
    "f1_score": round(f1, 3)
}])

pie2_combined_summary

In [ ]:
import os
import matplotlib.pyplot as plt

save_folder = "/content/PIE_results/video_0006_yolo_sam2_combined_with_annotations/"
os.makedirs(save_folder, exist_ok=True)

# Save CSV files
df_pie2_combined_annotation.to_csv(
    save_folder + "pie_video_0006_yolo_sam2_framewise_annotation_comparison.csv",
    index=False
)

pie2_combined_summary.to_csv(
    save_folder + "pie_video_0006_yolo_sam2_combined_summary.csv",
    index=False
)

# Clean table
pie2_combined_summary_clean = pie2_combined_summary.rename(columns={
    "dataset": "Dataset",
    "video": "Video",
    "start_frame": "Start frame",
    "sampled_frames": "Frames",
    "gt_annotations": "GT annotations",
    "yolo_detections": "YOLO detections",
    "true_positives": "TP",
    "false_positives": "FP",
    "false_negatives": "FN",
    "precision": "Precision",
    "recall": "Recall",
    "f1_score": "F1-score"
})

fig, ax = plt.subplots(figsize=(15, 2.5))
ax.axis("off")

table = ax.table(
    cellText=pie2_combined_summary_clean.values,
    colLabels=pie2_combined_summary_clean.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.15, 1.7)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight="bold")
    cell.set_linewidth(0.8)

image_path = save_folder + "pie_video_0006_yolo_sam2_combined_summary_table.png"

plt.savefig(image_path, bbox_inches="tight", dpi=300)
plt.show()

print("Saved files here:")
print(save_folder)
print("Saved table image:")
print(image_path)